# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My lane as an ML task

Task Type: Ranking

My lane is to rank anonymized search pages by how urgently they should be reviewed or refreshed. The model should help prioritize which pages come first in the work queue.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

Target / Proxy:
A page-level "needs review" signal, such as whether a page is likely to benefit from refresh based on its content and search performance signals.

If a direct label is not available, I would use a proxy label built from the repository’s refresh / review logic.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric:
Precision@K, especially Precision@50.

This fits the task because the output is a ranked queue, and the goal is to place the most useful pages near the top.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one content page with its search, traffic, engagement, and freshness features collected over recent time periods.eatures.

In [4]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Display the first few rows
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
print(df.shape)
df.info()
df.columns

(30000, 44)
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  str    
 1   client_id               30000 non-null  str    
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  str    
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  str    
 7   main_intent             27626 non-null  str    
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   str    
 11  model_used              24267 non-null  str    
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-nul

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='str')

In [7]:
import numpy as np

df["needs_review"] = (
    (df["trend_direction"] == "down") &
    (df["ctr"] < df["ctr"].median())
).astype(int)

df[["trend_direction", "ctr", "needs_review"]].head()

,trend_direction,ctr,needs_review
0,down,0.76,0
1,down,0.05,1
2,down,0.09,0
3,stable,0.49,0
4,down,0.13,0


### Target sketch

Since the dataset does not contain a ground-truth label, I created a simple proxy target called `needs_review`.

- `1` = the content should be reviewed.
- `0` = the content does not currently need review.

This is only a demonstration of what the target column could look like. In a real production system, the target would be defined using business rules or historical editorial decisions.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule would be too rigid, such as:
- if impressions are low, refresh the page
- if CTR is below a threshold, move it up

That misses interactions between many signals at once. ML can combine content, performance, and freshness features to learn which pages should be reviewed first, which is better than one hand-written threshold.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.